Here is the simple minimal form of a neural network using pytorch

In [12]:
!pip install torch
!pip install torchinfo

In [ ]:
import time 
import torch 
import torch.nn as nn 
torch.manual_seed(0)
class SimpleMlP(nn.Module):
    def __init__(self, input_size = 128, hidden_size = 516, output_size = 1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.reli2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, output_size)
    
    def forword(self, x):
        x = self.relu1(self.fc1(x))
        x = self.reli2(self.fc2(x))
        x = self.fc3(x)
        return x
model = SimpleMlP()

from custom_torchinfo import custom_summary
custom_summary(model, input_size(64,128))

This is how we generally calculate the size of a model. 

In [ ]:
model_prams = list(model.parameters()) + list(model.buffers()) # addition of all the model parameters like weights and bias and buffers. 
size = 0 
for t in model_prams: 
    size += t.numel() * t.element_size() # numel: number of elements, element_size: size of each element in a tensor. 
 

Calculate the latency

In [ ]:
model.eval() #starting model in evaluation mode  
start = time.perf_counter() # time.perf_counter is a standard python function to measure time. 
iters = 50 
with torch.inference_mode():
    for _ in range(iters): 
        _ = model(x) # we are not storing the output anywhere as we just need to check the latency not the quality in this case. 
        if x.device.type == "cuda": 
            torch.cuda.synchronize() # synchronizing the cuda just to get accurate calculation, as cpu generally be async. 

elapsed = time.perf_counter() - start
latency = elapsed / iters

In [ ]:
torch.cuda.reset_max_memory_cached()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats() #should reset the memroy stats before measuring or calculating 
model.eval() # starting model in inference eval 
with torch.inference_mode():
    y = model(x) #starting model run with x as input and storing y as output. 
torch.cuda.synchronize() 

current = torch.cuda.memory_allocated() / (1024 * 2 * 2)
preak = torch.cuda.max_memory_allocated() / (1024 * 2 * 2)


